# CFP Ranker '26

## Data Collection

### Team Collection

In [2]:
%pip install requests
%pip install pandas
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [18]:
import requests
import pandas as pd
import json
import time
from datetime import datetime

# --- CONFIG ---
YEAR = 2026  # Works for any year (e.g., 2023, 2024, 2025, 2026)
FBS_TEAMS_FILE = "./espn_fbs_teams.csv"

# --- ENDPOINTS & HEADERS ---
SCOREBOARD_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/scoreboard"
SUMMARY_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event="

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json"
}

In [ ]:
# NOTE: This block of code extracts the list of FBS teams from ESPN's API and saves it to a CSV file.
# This is arguably more useful because it includes team IDs and conference information

url = "https://site.web.api.espn.com/apis/v2/sports/football/college-football/standings?group=80"
# 80 is the ID for the FBS (Football Bowl Subdivision) in ESPN's API

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json"
}

response = requests.get(url, headers=headers)
response.raise_for_status()

data = response.json()

def parse_standings_node(node, parent_conference="", parent_conf_id=""):
    """Recursively walks standings nodes to extract teams, conference names, and conference IDs."""
    teams = []
    
    current_name = node.get('name', '')
    current_id = str(node.get('id', ''))
    
    # Prevent top-level league/subdivision labels from overriding actual conference names/IDs
    is_root_label = current_name.lower() in ['fbs', 'ncaa - football bowl subdivision', 'football bowl subdivision']
    
    if is_root_label:
        conf_label = ""
        conf_id = ""
    else:
        conf_label = parent_conference if parent_conference else current_name
        conf_id = parent_conf_id if parent_conf_id else current_id

    # Extract teams if standings entries exist at this level
    entries = node.get('standings', {}).get('entries', [])
    for entry in entries:
        team = entry.get('team', {})
        if team:
            teams.append({
                "team_id": team.get('id'),
                "display_name": team.get('displayName'),
                "short_name": team.get('shortDisplayName'),
                "conference": conf_label,
                "conference_id": conf_id
            })

    # Recurse into sub-children (e.g. East/West divisions)
    for child in node.get('children', []):
        teams.extend(parse_standings_node(child, parent_conference=conf_label, parent_conf_id=conf_id))

    return teams

fbs_teams = parse_standings_node(data)

df = pd.DataFrame(fbs_teams)
df = df.drop_duplicates(subset=['team_id']).reset_index(drop=True)

df.to_csv("espn_fbs_teams.csv", index=False)
print(f"Successfully extracted {len(df)} FBS teams with conference IDs.")

Successfully extracted 138 FBS teams with conference IDs.


## Retrieve A Single Game

In [16]:
# --- CONFIG ---
YEAR = 2026  # Works for any year (e.g., 2023, 2024, 2025, 2026)
FBS_TEAMS_FILE = "../data/espn_fbs_teams.csv"

# --- ENDPOINTS & HEADERS ---
SCOREBOARD_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/scoreboard"
SUMMARY_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event="

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json"
}

params = {
    "dates": "20261017",
    "groups": "80",
    "limit": "300"
}

response = requests.get(SCOREBOARD_URL, headers=HEADERS, params=params)
target_game = None

try:
	games = response.json().get("events", [])
	if response.status_code == 200:
		for event in games:
			game_name = event.get("name", "")
			if "Alabama" in game_name and "Tennessee" in game_name:
				target_game = event
				break
except Exception as e:
	print(f"Error fetching games: {e}")
	games = []

if target_game:
	print(f"Target game found: {target_game.get('name', '')}")
else:
	print("Target game not found.")

game_id = target_game.get("id", "") if target_game else None
if game_id:
	print(f"Game ID: {game_id}")
else:
	print("Game ID not found.")

def export_boxscore_to_csv(game_id, output_filename="boxscore.csv"):
    # Base URL for the summary endpoint
    summary_url = f"https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event={game_id}"
    
    headers = HEADERS

    print(f"Fetching summary data for Game ID: {game_id}...")
    response = requests.get(summary_url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        
        # 2. Navigate to the Team Boxscore section
        teams_data = data.get("boxscore", {}).get("teams", [])
        
        if not teams_data:
            print("No box score data available yet.")
            return

        # 3. Parse the data into a list of dictionaries
        rows = []
        for team in teams_data:
            team_name = team.get("team", {}).get("displayName", "Unknown")
            stats = team.get("statistics", [])
            
            # Create a dictionary for the row, starting with the team name
            team_row = {"Team": team_name}
            
            # Loop through each stat (e.g., 'totalYards', 'firstDowns') and add it to the row
            for stat in stats:
                stat_name = stat.get("label", stat.get("name"))
                stat_value = stat.get("displayValue")
                team_row[stat_name] = stat_value
                
            rows.append(team_row)

        # 4. Use Pandas to convert the rows to a DataFrame and export to CSV
        df = pd.DataFrame(rows)
        
        # Optional: Transpose the DataFrame if you want the stats as rows instead of columns
        df = df.set_index("Team").T
        df.reset_index(inplace=True)
        df.rename(columns={"index": "Stat Category"}, inplace=True)
        
        df.to_csv(output_filename, index=False)
        print(f"\nSuccess! Data exported to {output_filename}")
        print(df.head())
        
    else:
        print(f"Failed to retrieve data. Status Code: {response.status_code}")

# Use the game ID we found in the previous step
TARGET_GAME_ID = game_id # Replace with your dynamic Game ID
export_boxscore_to_csv(TARGET_GAME_ID, f"Tennessee_Alabama_Boxscore_{TARGET_GAME_ID}.csv")

def save_full_game_json(game_id, output_filename="full_game_data.json"):
    summary_url = f"https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event={game_id}"
    
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json"
    }

    print(f"Fetching full summary JSON for Game ID: {game_id}...")
    response = requests.get(summary_url, headers=headers)

    if response.status_code == 200:
        # Parse the JSON response
        data = response.json()
        
        # Open a new file in write mode and dump the JSON into it
        # indent=4 makes the file highly readable (pretty-printed)
        with open(output_filename, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=4)
            
        print(f"Success! The full JSON payload has been saved to '{output_filename}'.")
    else:
        print(f"Failed to retrieve data. Status Code: {response.status_code}")

# The Game ID for your specific matchup
TARGET_GAME_ID = game_id

save_full_game_json(TARGET_GAME_ID, "Tennessee_Alabama_Full_Summary.json")

Target game found: Alabama Crimson Tide at Tennessee Volunteers
Game ID: 401856723
Fetching summary data for Game ID: 401856723...

Success! Data exported to Tennessee_Alabama_Boxscore_401856723.csv
Team        Stat Category Alabama Crimson Tide Tennessee Volunteers
0               1st Downs                   12                   20
1     3rd down efficiency                 2-13                 6-15
2     4th down efficiency                  1-3                  1-1
3             Total Yards                  201                  347
4                 Passing                  147                  197
Fetching full summary JSON for Game ID: 401856723...
Success! The full JSON payload has been saved to 'Tennessee_Alabama_Full_Summary.json'.


In [ ]:
def discover_logged_statistics(input_filename="Tennessee_Alabama_Full_Summary.json", output_filename="logged_statistics.txt"):
    num_team_stats = 0
    num_player_stats = 0
    # Load the JSON data
    with open(input_filename, "r", encoding="utf-8") as file:
        data = json.load(file)
        
    # Open the output text file in write mode
    with open(output_filename, "w", encoding="utf-8") as out_file:
        out_file.write("=== DISCOVERING LOGGED STATISTICS ===\n\n")
        
        boxscore = data.get("boxscore", {})
        
        # 1. Extract Team-Level Statistics
        out_file.write("--- TEAM STATISTICS TRACKED ---\n")
        teams = boxscore.get("teams", [])
        if teams:
            team_stats = teams[0].get("statistics", [])
            for stat in team_stats:
                stat_name = stat.get("name", "Unknown key")
                stat_label = stat.get("label", "Unknown label")
                out_file.write(f"- {stat_label} (Key: {stat_name})\n")
                num_team_stats += 1
        else:
            out_file.write("No team statistics found.\n")
            
        # 2. Extract Individual Player Stat Categories
        out_file.write("\n--- INDIVIDUAL PLAYER STAT CATEGORIES ---\n")
        players = boxscore.get("players", [])
        if players:
            player_categories = players[0].get("statistics", [])
            for category in player_categories:
                cat_name = category.get("name", "Unknown")
                out_file.write(f"\n[{cat_name.upper()}]\n")
                
                keys = category.get("keys", [])
                labels = category.get("labels", [])
                
                for key, label in zip(keys, labels):
                    out_file.write(f"  - {label} (Key: {key})\n")
                    num_player_stats += 1
        else:
            out_file.write("No player statistics found.\n")

    # Print a single confirmation message to the console
    print(f"Success! The full list of statistics has been saved to '{output_filename}'.")
    print(f"Total team statistics tracked: {num_team_stats}")
    print(f"Total individual player stat categories: {num_player_stats}")
    
discover_logged_statistics("Tennessee_Alabama_Full_Summary.json")

Success! The full list of statistics has been saved to 'logged_statistics.txt'.
Total team statistics tracked: 15
Total individual player stat categories: 50


# Moving Forward

## Plan A

Find the missing values, document that, and determine what ESPN stat loggers deam as most important

## Plan B

If there are no missing values, find ways to use the data to make conjecture about the strength of teams based on their game-by-game stats.  Could be worth trying to find that play-by-play endpoint

### Retrieve All The Games For The 2026 Season

In [ ]:
# Retrieve all FBS games for the 2026 season and save them to a CSV file

start_date = pd.Timestamp("2026-08-01")
end_date = pd.Timestamp("2027-02-01")

rows = []
chunk_start = start_date

while chunk_start <= end_date:
	response = requests.get(
		SCOREBOARD_URL,
		headers=HEADERS,
		params={
			"dates": f"{chunk_start:%Y%m%d}",
			"groups": "80",
			"limit": "500",
		},
		timeout=30,
	)
	response.raise_for_status()

	for event in response.json().get("events", []):
		competition = (event.get("competitions") or [{}])[0]
		competitors = competition.get("competitors", [])
		home = next((team for team in competitors if team.get("homeAway") == "home"), {})
		away = next((team for team in competitors if team.get("homeAway") == "away"), {})

		rows.append({
			"event_id": event.get("id"),
			"date": event.get("date"),
			"name": event.get("name"),
			"week": event.get("week", {}).get("number"),
			"home_team": home.get("team", {}).get("displayName"),
			"home_team_id": home.get("team", {}).get("id"),
			"home_score": home.get("score"),
			"away_team": away.get("team", {}).get("displayName"),
			"away_team_id": away.get("team", {}).get("id"),
			"away_score": away.get("score"),
			"status": event.get("status", {}).get("type", {}).get("description"),
			"venue": competition.get("venue", {}).get("fullName"),
			"neutral_site": competition.get("neutralSite"),
		})

	chunk_start += pd.Timedelta(days=1)
	time.sleep(0.2)

season_games_df = (
	pd.DataFrame(rows)
	.drop_duplicates(subset="event_id")
	.sort_values("date")
	.reset_index(drop=True)
)

output_file = "espn_2026_fbs_season_games.csv"
season_games_df.to_csv(output_file, index=False)

print(f"Retrieved {len(season_games_df)} games and saved them to {output_file}.")
season_games_df.head()

Retrieved 950 games and saved them to espn_2026_fbs_season_games.csv.


,event_id,date,name,week,home_team,home_team_id,home_score,away_team,away_team_id,away_score,status,venue,neutral_site
0,401856766,2026-08-29T16:00Z,North Carolina Tar Heels at TCU Horned Frogs,1,TCU Horned Frogs,2628,10,North Carolina Tar Heels,153,15,Final,Aviva Stadium,True
1,401864494,2026-08-29T19:00Z,San José State Spartans at USC Trojans,1,USC Trojans,30,42,San José State Spartans,23,26,Final,Los Angeles Memorial Coliseum,False
2,401858202,2026-08-29T19:30Z,NC State Wolfpack at Virginia Cavaliers,1,Virginia Cavaliers,258,34,NC State Wolfpack,152,8,Final,Scott Stadium,False
3,401864577,2026-08-29T21:30Z,Jacksonville State Gamecocks at North Dakota S...,1,North Dakota State Bison,2449,33,Jacksonville State Gamecocks,55,7,Final,Fargodome,False
4,401866408,2026-08-29T22:30Z,Sacramento State Hornets at Eastern Michigan E...,1,Eastern Michigan Eagles,2199,28,Sacramento State Hornets,16,17,Final,Rynearson Stadium,False


### Check for Missing Stats

In [ ]:
season_games_df = pd.read_csv("espn_2026_fbs_season_games.csv")

# For every game, we want to check if there are any missing values among the 65 statistical categories 
# that ESPN tracks for each game. If any are missing, we will log the game ID and the missing categories to a text file.
with open("missing_stats.log", "w") as f:
	for index, row in season_games_df.iterrows():
		if row["status"] == "Final":
			game_id = row["event_id"]
			summary_url = f"https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event={game_id}"
			
			response = requests.get(summary_url, headers=HEADERS)
			if response.status_code != 200:
				f.write(f"Game ID {game_id}: Failed to retrieve data (Status Code: {response.status_code})\n")
				continue
			
			data = response.json()
			boxscore = data.get("boxscore", {})
			
			# Check team statistics
			teams = boxscore.get("teams", [])
			for team in teams:
				stats = team.get("statistics", [])
				for stat in stats:
					if stat.get("displayValue") is None:
						f.write(f"Game ID {game_id}: Missing team stat '{stat.get('label')}'\n")
			
			# Check player statistics
			players = boxscore.get("players", [])
			for player in players:
				categories = player.get("statistics", [])
				for category in categories:
					keys = category.get("keys", [])
					labels = category.get("labels", [])
					values = category.get("values", [])
					for key, label, value in zip(keys, labels, values):
						if value is None:
							f.write(f"Game ID {game_id}: Missing player stat '{label}' (Key: {key})\n")

with open("missing_stats.log", "r") as f:
	content = f.read().strip()
	if not content:
		print("No missing statistics found for any game.")
	else:
		print("Missing statistics were found. Please check 'missing_stats.log' for details.")

No missing statistics found for any game.


## Execute Plan B

An interesting thing to note is that the team stats provided are mostly offensive statistics.  These can be used to make conjecture about team defenses though.  If you were to aggregate a team’s opponents’ offensive stats, you could use that to make conjecture about their defense.

Additionally, you could compare how that opposing team did in that game versus in other games to make conjecture about a team's defense.

*i.e. Alabama had a high flying offense in their previous games but was shut down by both Georgia’s and Tennessee’s defenses.  Statistically, that shows across various categories that are measured in the API.  We can use that to make conjecture about Alabama’s offense, but also Georgia’s and Tennessee’s defenses.*

That information can be used to neutralize statistical outliers or upset games in a ranking algorithm.

In [ ]:
# As an example, let's retrieve the box scores for all games played by Tennessee and their opponents.
# We can store this data in a CSV file for further analysis.
import requests
import pandas as pd
import json
import time
from datetime import datetime

# --- CONFIG ---
YEAR = 2026  # Works for any year (e.g., 2023, 2024, 2025, 2026)
FBS_TEAMS_FILE = "./espn_fbs_teams.csv"

# --- ENDPOINTS & HEADERS ---
SCOREBOARD_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/scoreboard"
SUMMARY_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event="

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json"
}

# Rewrite this to store the data in a dataframe
def retrieve_boxscore(game_id):
	summary_url = f"https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event={game_id}"
	
	response = requests.get(summary_url, headers=HEADERS)
	if response.status_code != 200:
		print(f"Failed to retrieve data for Game ID {game_id}. Status Code: {response.status_code}")
		return None
	
	data = response.json()
	boxscore = data.get("boxscore", {})
	
	# Extract team statistics
	teams_data = boxscore.get("teams", [])
	team_stats_list = []
	
	for team in teams_data:
		team_name = team.get("team", {}).get("displayName", "Unknown")
		stats = team.get("statistics", [])
		
		for stat in stats:
			stat_name = stat.get("name", stat.get("name"))
			stat_value = stat.get("value")
			if stat_value == '-':
				stat_value = stat.get("displayValue")
			team_stats_list.append({
				"Game ID": game_id,
				"Team": team_name,
				"Stat Category": stat_name,
				"Stat Value": stat_value
			})
	
	return pd.DataFrame(team_stats_list)
	
fbs_teams_df = pd.read_csv(FBS_TEAMS_FILE)
tn_team_id = fbs_teams_df.loc[fbs_teams_df['display_name'] == 'Tennessee Volunteers', 'team_id'].values[0]

season_games_df = pd.read_csv("espn_2026_fbs_season_games.csv")
# Filter for games where Tennessee is either the home or away team
tn_games_df = season_games_df[((season_games_df['home_team_id'] == tn_team_id) | (season_games_df['away_team_id'] == tn_team_id)) & (season_games_df['status'] == 'Final')]

for index, row in tn_games_df.iterrows():
	game_id = row['event_id']
	boxscore_df = retrieve_boxscore(game_id)
	if boxscore_df is not None:
		output_filename = f"./data/Tennessee_Game_{game_id}_Boxscore.csv"
		boxscore_df.to_csv(output_filename, index=False)